In [1]:
# Configuracion principal
N_ITER = 4
CV = 2
RANDOM_STATE = 42
TARGET_CLASS = 1
TARGETS = ("1_vs_resto", "5_vs_resto")
SEARCH_N_JOBS = 1


In [2]:
from __future__ import annotations

import json
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.base import clone
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_recall_curve,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, label_binarize

REPO_ROOT = Path(".").resolve()
INPUT_BASE = REPO_ROOT / "data" / "intermediate" / "05_seleccion_v2"
PRE_RUN_PATH = (
    REPO_ROOT
    / "reports"
    / "10_modelo_redmlp_v2"
    / "20260505"
    / "RedMLP_v2_01_2026-05-05"
)
OUTPUT_BASE = REPO_ROOT / "reports" / "12_tuning_redmlp_v2"


def latest_input_path() -> Path:
    candidates = sorted([p for p in INPUT_BASE.iterdir() if p.is_dir()])
    if not candidates:
        raise FileNotFoundError(f"No hay subdirectorios en {INPUT_BASE}")
    return candidates[-1]


def resolve_target_class(y: pd.Series, target: int) -> int:
    classes = list(pd.Series(y).unique())
    if target in classes:
        return target
    for klass in classes:
        if str(klass) == str(target):
            return klass
    raise ValueError(f"TARGET_CLASS {target} no esta en clases disponibles: {classes}")


def build_balancer(balance_name: str):
    if balance_name == "NONE":
        return None
    if balance_name == "SMOTE":
        return SMOTE(random_state=RANDOM_STATE)
    if balance_name == "UNDER":
        return RandomUnderSampler(random_state=RANDOM_STATE)
    if balance_name == "SMOTEENN":
        return SMOTEENN(random_state=RANDOM_STATE)
    raise ValueError(f"Balanceo no soportado: {balance_name}")


def build_mlp(num_classes: int, **overrides) -> MLPClassifier:
    params = dict(
        hidden_layer_sizes=(96, 48) if num_classes > 2 else (64, 32),
        activation="relu",
        solver="adam",
        learning_rate_init=0.002,
        max_iter=90,
        batch_size=128,
        alpha=1e-4,
        early_stopping=True,
        n_iter_no_change=6,
        validation_fraction=0.12,
        shuffle=True,
        random_state=RANDOM_STATE,
        verbose=False,
    )
    params.update(overrides)
    return MLPClassifier(**params)


def build_pipeline(balance_name: str, num_classes: int) -> Pipeline:
    steps = []
    balancer = build_balancer(balance_name)
    if balancer is not None:
        steps.append(("balance", clone(balancer)))
    steps.append(("scale", StandardScaler(with_mean=False)))
    steps.append(("model", build_mlp(num_classes)))
    return Pipeline(steps)


def choose_best_candidate(target_name: str) -> dict:
    summary_csv = PRE_RUN_PATH / "resumen_modelos_redmlp_global.csv"
    if not summary_csv.exists():
        raise FileNotFoundError(f"No existe resumen global: {summary_csv}")

    df = pd.read_csv(summary_csv)
    required = {"target", "modelo", "balanceo", "cv_recall_target", "cv_macro_f1"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Faltan columnas en {summary_csv}: {sorted(missing)}")

    candidates = df.dropna(subset=["modelo", "balanceo", "cv_recall_target", "cv_macro_f1"]).copy()
    candidates = candidates[candidates["target"].astype(str) == target_name]
    if candidates.empty:
        raise ValueError(f"No hay candidatos validos para {target_name}")

    best = candidates.sort_values(
        ["cv_recall_target", "cv_macro_f1"], ascending=False
    ).iloc[0]
    return {
        "target": target_name,
        "base_name": str(best["modelo"]),
        "balance_name": str(best["balanceo"]),
        "cv_recall_target_pre": float(best["cv_recall_target"]),
        "cv_macro_f1_pre": float(best["cv_macro_f1"]),
    }


def save_report_pdf(
    pdf_path: Path,
    target_name: str,
    y_test: pd.Series,
    y_test_adj: pd.Series,
    y_pred_test: np.ndarray,
    y_proba: np.ndarray,
    class_order: np.ndarray,
    class_order_adj: np.ndarray,
) -> None:
    cm = confusion_matrix(y_test, y_pred_test, labels=class_order)
    with PdfPages(pdf_path) as pdf:
        ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_order).plot(cmap="Blues")
        plt.title(f"Matriz de Confusion - {target_name}")
        pdf.savefig()
        plt.close()

        if np.unique(y_test).size < 2:
            return

        y_bin = label_binarize(y_test_adj, classes=class_order_adj)
        if y_bin.shape[1] == 1:
            y_bin = np.hstack([1 - y_bin, y_bin])

        proba_for_curves = y_proba
        if proba_for_curves.shape[1] == 1:
            proba_for_curves = np.hstack([1 - y_proba, y_proba])

        plt.figure()
        for i, klass in enumerate(class_order):
            fpr, tpr, _ = roc_curve(y_bin[:, i], proba_for_curves[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f"Clase {klass} (AUC={roc_auc:.2f})")
        plt.plot([0, 1], [0, 1], "k--")
        plt.title(f"Curvas ROC por clase - {target_name}")
        plt.xlabel("FPR")
        plt.ylabel("TPR")
        plt.legend()
        pdf.savefig()
        plt.close()

        plt.figure()
        for i, klass in enumerate(class_order):
            precision, recall, _ = precision_recall_curve(y_bin[:, i], proba_for_curves[:, i])
            pr_auc = average_precision_score(y_bin[:, i], proba_for_curves[:, i])
            plt.plot(recall, precision, label=f"Clase {klass} (PR AUC={pr_auc:.2f})")
        plt.title(f"Curvas Precision-Recall por clase - {target_name}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.legend()
        pdf.savefig()
        plt.close()


def tune_candidate(candidate: dict, input_path: Path, output_dir: Path) -> dict:
    target_name = candidate["target"]
    base_name = candidate["base_name"]
    balance_name = candidate["balance_name"]

    target_input_path = input_path / target_name
    if not target_input_path.exists():
        raise FileNotFoundError(f"No existe input para {target_name}: {target_input_path}")

    print(f"\n=== Tuning {target_name} | {base_name} [{balance_name}] ===")

    X_train = pd.read_parquet(target_input_path / f"X_train_{base_name}.parquet").astype("float32")
    X_test = pd.read_parquet(target_input_path / f"X_test_{base_name}.parquet").astype("float32")
    y_train = pd.read_parquet(target_input_path / f"y_train_{base_name}.parquet").squeeze()
    y_test = pd.read_parquet(target_input_path / f"y_test_{base_name}.parquet").squeeze()

    target_class = resolve_target_class(y_train, TARGET_CLASS)
    y_min = int(pd.Series(y_train).min())
    y_train_adj = y_train - y_min
    y_test_adj = y_test - y_min
    target_class_adj = target_class - y_min
    num_classes = int(pd.Series(y_train_adj).nunique())

    pipeline = build_pipeline(balance_name, num_classes)
    scorers = {
        "recall_target": make_scorer(
            recall_score, labels=[target_class_adj], average="macro", zero_division=0
        ),
        "f1_macro": "f1_macro",
    }
    param_distributions = {
        "model__hidden_layer_sizes": [(64, 32), (96, 48), (128, 64), (128, 64, 32)],
        "model__learning_rate_init": [0.0008, 0.001, 0.002, 0.003],
        "model__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
        "model__batch_size": [128, 256, 512],
        "model__max_iter": [80, 120],
    }

    cv = StratifiedKFold(n_splits=CV, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=N_ITER,
        scoring=scorers,
        refit="recall_target",
        cv=cv,
        n_jobs=SEARCH_N_JOBS,
        pre_dispatch=1,
        random_state=RANDOM_STATE,
        verbose=2,
        return_train_score=True,
    )

    start = time.time()
    search.fit(X_train, y_train_adj)
    tiempo_min = (time.time() - start) / 60

    best_model = search.best_estimator_
    y_pred_train_adj = best_model.predict(X_train)
    y_pred_test_adj = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)

    y_pred_train = y_pred_train_adj + y_min
    y_pred_test = y_pred_test_adj + y_min
    class_order_adj = best_model.named_steps["model"].classes_
    class_order = class_order_adj + y_min

    report_test = pd.DataFrame(
        classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)
    ).T
    report_train = pd.DataFrame(
        classification_report(y_train, y_pred_train, output_dict=True, zero_division=0)
    ).T

    target_output_dir = output_dir / target_name / f"best_{target_name}"
    target_output_dir.mkdir(parents=True, exist_ok=True)

    cv_results = pd.DataFrame(search.cv_results_)
    cv_results.to_csv(target_output_dir / f"cv_results_{base_name}.csv", index=False)

    report_test["set"] = "test"
    report_train["set"] = "train"
    full_report = pd.concat([report_train, report_test])
    full_report["tiempo_min"] = tiempo_min
    full_report["target"] = target_name
    full_report["modelo_base"] = base_name
    full_report["balanceo"] = balance_name
    full_report.to_csv(target_output_dir / f"metricas_{base_name}_TUNED.csv")

    save_report_pdf(
        target_output_dir / f"reporte_{base_name}_TUNED.pdf",
        target_name,
        y_test,
        y_test_adj,
        y_pred_test,
        y_proba,
        class_order,
        class_order_adj,
    )

    best_params = search.best_params_
    with open(target_output_dir / f"best_params_{base_name}.json", "w", encoding="utf-8") as f:
        json.dump(best_params, f, indent=2, ensure_ascii=False)

    summary = {
        "target": target_name,
        "modelo_base": base_name,
        "balanceo": balance_name,
        "target_class": target_class,
        "target_class_adj": target_class_adj,
        "pre_cv_recall_target": candidate["cv_recall_target_pre"],
        "pre_cv_macro_f1": candidate["cv_macro_f1_pre"],
        "tuned_cv_recall_target": float(search.best_score_),
        "tuned_cv_macro_f1": float(
            cv_results.loc[search.best_index_, "mean_test_f1_macro"]
        ),
        "accuracy_test": accuracy_score(y_test, y_pred_test),
        "macro_f1_test": f1_score(y_test, y_pred_test, average="macro", zero_division=0),
        "recall_target_test": recall_score(
            y_test, y_pred_test, labels=[target_class], average="macro", zero_division=0
        ),
        "accuracy_train": accuracy_score(y_train, y_pred_train),
        "macro_f1_train": f1_score(y_train, y_pred_train, average="macro", zero_division=0),
        "tiempo_min": tiempo_min,
        "best_params": json.dumps(best_params, ensure_ascii=False),
    }
    pd.DataFrame([summary]).to_csv(
        target_output_dir / f"resumen_tuning_{target_name}.csv", index=False
    )

    print(
        f"BEST tuned {target_name}: recall_cv={summary['tuned_cv_recall_target']:.3f}, "
        f"macro_f1_test={summary['macro_f1_test']:.3f}"
    )
    return summary


def main() -> None:
    input_path = latest_input_path()
    if not PRE_RUN_PATH.exists():
        raise FileNotFoundError(f"No existe la corrida previa: {PRE_RUN_PATH}")

    run_id = datetime.today().strftime("%Y%m%d")
    output_dir = OUTPUT_BASE / run_id / f"RedMLP_tuning_v2_{datetime.today().strftime('%Y-%m-%d')}"
    output_dir.mkdir(parents=True, exist_ok=True)

    candidates = [choose_best_candidate(target) for target in TARGETS]
    print("Candidatos seleccionados:")
    for candidate in candidates:
        print(
            f"  {candidate['target']}: {candidate['base_name']} "
            f"[{candidate['balance_name']}] "
            f"pre_cv_recall={candidate['cv_recall_target_pre']:.3f}"
        )

    summaries = [tune_candidate(candidate, input_path, output_dir) for candidate in candidates]
    pd.DataFrame(summaries).to_csv(output_dir / "resumen_tuning_redmlp_v2.csv", index=False)
    print(f"\nResumen global guardado en: {output_dir / 'resumen_tuning_redmlp_v2.csv'}")


In [3]:
main()


Candidatos seleccionados:
  1_vs_resto: MaxAbs_FE_ALL [SMOTEENN] pre_cv_recall=0.883
  5_vs_resto: MaxAbs_FE_ALL [SMOTEENN] pre_cv_recall=0.950

=== Tuning 1_vs_resto | MaxAbs_FE_ALL [SMOTEENN] ===
Fitting 2 folds for each of 4 candidates, totalling 8 fits
[CV] END model__alpha=0.001, model__batch_size=512, model__hidden_layer_sizes=(96, 48), model__learning_rate_init=0.002, model__max_iter=80; total time=67.4min
[CV] END model__alpha=0.001, model__batch_size=512, model__hidden_layer_sizes=(96, 48), model__learning_rate_init=0.002, model__max_iter=80; total time=66.3min
[CV] END model__alpha=0.001, model__batch_size=256, model__hidden_layer_sizes=(128, 64, 32), model__learning_rate_init=0.001, model__max_iter=80; total time=73.8min
[CV] END model__alpha=0.001, model__batch_size=256, model__hidden_layer_sizes=(128, 64, 32), model__learning_rate_init=0.001, model__max_iter=80; total time=73.3min
[CV] END model__alpha=0.01, model__batch_size=512, model__hidden_layer_sizes=(64, 32), model_